# Lesson 18: ਕ੍ਰਿਪਟੋਗ੍ਰਾਫਿਕ ਰਸੀਦਾਂ ਨਾਲ AI ਏਜੰਟਾਂ ਨੂੰ ਸੁਰੱਖਿਅਤ ਕਰਨਾ

## ਹੈਂਡਸ-ਆਨ ਨੋਟਬੁੱਕ

ਇਹ ਨੋਟਬੁੱਕ ਚਾਰ ਅਭਿਆਸਾਂ ਵਿੱਚੋਂ ਗੁਜ਼ਰਦਾ ਹੈ:

1. **ਆਪਣੀ ਪਹਿਲੀ ਰਸੀਦ ਸਾਈਨ ਕਰੋ** ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ ਅਤੇ ਇਸ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
2. **ਰਸੀਦ ਵਿੱਚ ਤਬਦੀਲੀ ਕਰੋ** ਅਤੇ ਵੈਰੀਫਿਕੇਸ਼ਨ ਦੀ ਅਸਫਲਤਾ ਨੂੰ ਦੇਖੋ।
3. **ਤਿੰਨ-ਰਸੀਦਾਂ ਦੀ ਚੇਨ ਬਣਾਓ** ਅਤੇ ਚੇਨ ਦੀ ਅਖੰਡਤਾ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
4. **ਮਾਈਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਟੂਲ ਕਾਲ ਨੂੰ ਰੈਪ ਕਰੋ** ਤਾਂ ਜੋ ਹਰ ਕ੍ਰਿਆ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਕਰੇ।

ਸਾਰੇ ਕ੍ਰਿਪਟੋਗ੍ਰਾਫਿਕ ਪ੍ਰਿਮਿਟਿਵਜ਼ ਚੰਗੇ ਤਰੀਕੇ ਨਾਲ ਸਾਂਭੀਆਂ ਲਾਇਬ੍ਰੇਰੀਆਂ ਤੋਂ ਆਏ ਹਨ (`pynacl` Ed25519 ਲਈ, `jcs` RFC 8785 ਕੈਨੋਨਿਕਲ JSON ਲਈ, Python ਦੇ ਸਟੈਂਡਰਡ ਲਾਇਬ੍ਰੇਰੀ ਤੋਂ `hashlib` SHA-256 ਲਈ)। ਰਸੀਦ ਲੌਜਿਕ ਖੁਦ ਸਾਫ਼ ਸਧਾਰਨ Python ਹੈ ਜਿਸਨੂੰ ਤੁਸੀਂ ਪੜ੍ਹ ਅਤੇ ਸੋਧ ਸਕਦੇ ਹੋ।

ਸੈੱਲਾਂ ਨੂੰ ਕ੍ਰਮ ਵਿੱਚ ਚਲਾਓ। ਹਰ ਸੈਕਸ਼ਨ ਛੋਟਾ ਅਤੇ ਸੁਤੰਤਰ ਹੈ।


## Setup

ਦੋਹਾਂ dependencies ਇੰਸਟਾਲ ਕਰੋ। ਦੋਹਾਂ ਦੇ ਕੋਡ ਖੁੱਲ੍ਹੇ ਲਾਇਸੰਸ (Apache-2.0 / MIT) ਦੇ ਹਨ।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ਮਦਦਗਾਰ ਉਪਯੋਗਿਤਾ

ਇਹ ਦੋ ਮਦਦਗਾਰਾਂ ਬਿਨਾਂ ਪੈਡਿੰਗ ਦੇ base64url ਇੰਕੋਡਿੰਗ ਅਤੇ ਅਣਿਰਧਾਰਿਤ ਵਸਤਾਂ ਦੀ SHA-256 ਹੈਸ਼ਿੰਗ ਨੂੰ ਹੰਢਾਉਂਦੇ ਹਨ। ਇਹ ਨੋਟਬੁੱਕ ਦੇ ਬਾਕੀ ਹਿੱਸੇ ਨੂੰ ਸਵীਕਾਰ ਲਾਜਿਕ 'ਤੇ ਧਿਆਨ ਕੇਂਦਰਿਤ ਰੱਖਦੇ ਹਨ।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: ਆਪਣੇ ਪਹਿਲੇ ਰਸੀਦ 'ਤੇ ਦਸਤਖਤ ਕਰੋ

ਕਲਪਨਾ ਕਰੋ ਕਿ ਸਾਡੇ ਏਜੰਟ ਲਈ **ਕੌਂਟੋਸੋ ਟ੍ਰੈਵਲ** ਨੇ ਹੁਣੇ ही ਸਿਡਨੀ ਤੋਂ ਲੋਸ ਐਂਜਲਸ ਲਈ ਇੱਕ ਗਾਹਕ ਲਈ ਉਡਾਣਾਂ ਦੀ ਖੋਜ ਕੀਤੀ ਹੈ। ਅਸੀਂ ਇਸ ਟੂਲ ਕਾਲ ਨੂੰ ਇੱਕ ਸਾਇਨ ਕੀਤੀ ਰਸੀਦ ਵੱਜੋਂ ਦਰਜ ਕਰਨਾ ਚਾਹੁੰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਭਵਿੱਖ ਵਿੱਚ ਕੋਈ ਆਡੀਟਰ ਬਿਨਾ ਸਾਡੇ 'ਤੇ ਭਰੋਸਾ ਕੀਤਾ ਬਿਨਾ ਇਸ ਦੀ ਪੁਸ਼ਟੀ ਕਰ ਸਕੇ।

### ਕਦਮ 1.1: ਸਾਇਨਿੰਗ ਕੀ ਤਿਆਰ ਕਰੋ

ਉਤਪਾਦਨ ਵਿੱਚ, ਏਜੰਟ ਦੀ ਸਾਇਨਿੰਗ ਕੀ ਇੱਕ ਹਾਰਡਵੇਅਰ ਸੁਰੱਖਿਆ ਮੋਡੀਊਲ (HSM), ਅਜ਼ੂਰ ਕੀ ਵਾਲਟ ਜਾਂ ਕਿਸੇ ਸਮਾਨ ਸੁਰੱਖਿਅਤ ਸਟੋਰ ਵਿੱਚ ਰਹਿੰਦੀ। ਇਸ ਪੈਠਕ ਲਈ ਅਸੀਂ ਮੈਮੋਰੀ ਵਿੱਚ ਇੱਕ ਤਾਜ਼ਾ ਕੀ ਬਣਾਉਂਦੇ ਹਾਂ।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: ਰਸੀਦ ਪੇਲੋਡ ਤਿਆਰ ਕਰੋ

ਪੇਲੋਡ ਵਿੱਚ ਉਹ ਸਭ ਕੁਝ ਸ਼ਾਮਲ ਹੁੰਦਾ ਹੈ ਜੋ ਅਸੀਂ ਚਾਹੁੰਦੇ ਹਾਂ ਕਿ ਰਸੀਦ ਅਟੈਸਟ ਕਰੇ: ਕਿਸਨੇ ਕੰਮ ਕੀਤਾ, ਕਿਹੜਾ ਟੂਲ, ਕਿਹੜੇ ਦਲੀਲਾਂ ਨਾਲ, ਕੀ ਵਾਪਸ ਮਿਲਿਆ, ਕਿਸ ਨੀਤੀ ਦੇ ਅਧੀਨ, ਅਤੇ ਕਦੋਂ। ਅਸੀਂ ਦਲੀਲਾਂ ਅਤੇ ਨਤੀਜੇ ਦਾ ਹੈਸ਼ ਕਰਦੇ ਹਾਂ ਬਜਾਏ ਇਸ ਨੂੰ ਲਾਈਨ ਵਿੱਚ ਸ਼ਾਮਲ ਕਰਨ ਦੇ ਤਾਂ ਜੋ ਰਸੀਦ ਸੰਵੇਦਨਸ਼ੀਲ ਸਮੱਗਰੀ ਨੂੰ ਲੀਕ ਨਾ ਕਰੇ।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ਕਦਮ 1.3: ਰਸੀਦ 'ਤੇ ਦਸਤਖਤ ਕਰੋ ਅਤੇ ਇਕੱਠਾ ਕਰੋ

ਤਿੰਨ ਕਦਮ:

1. ਦੋਵੇਂ ਇੰਪਲੀਮੇਂਟੇਸ਼ਨਾਂ ਵੱਲੋਂ ਇੱਕੋ ਲਾਜ਼ਮੀ ਰਸੀਦ ਬਣਾਉਣ ਲਈ ਪੇਲੋਡ ਨੂੰ JCS ਦੀ ਵਰਤੋਂ ਨਾਲ ਕੈਨੋਨੀਕਲਾਈਜ਼ ਕਰੋ ਤਾਂ ਜੋ ਬਾਈਟ-ਮਿਲਦੇਜ਼ਿਲਦੇ ਬਾਈਟ ਬਣਨ।
2. ਕੈਨੋਨੀਕਲ ਬਾਈਟਾਂ ਦਾ SHA-256 ਨਾਲ ਹੈਸ਼ ਬਣਾਓ।
3. Ed25519 ਪ੍ਰਾਈਵੇਟ ਕੀ ਨਾਲ ਹੈਸ਼ 'ਤੇ ਦਸਤਖਤ ਕਰੋ।

ਦਸਤਖਤ ਫਿਰ ਅਸਲ ਪੇਲੋਡ ਨਾਲ ਲਗਾਈ ਜਾਂਦੀ ਹੈ ਤਾਂ ਜੋ ਆਖਰੀ ਰਸੀਦ ਬਣੇ।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: ਰਸੀਦ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ

ਪੁਸ਼ਟੀਕਰਨ ਪ੍ਰਕਿਰਿਆ ਨੂੰ ਉਲਟ ਦਿੰਦਾ ਹੈ। ਅਸੀਂ ਦਸਤਖ਼ਤ ਨੂੰ ਹਟਾ ਕੇ, canonical ਹੈਸ਼ ਨੂੰ ਦੁਬਾਰਾ ਗਣਨਾ ਕਰਦੇ ਹਾਂ, ਅਤੇ ਰਸੀਦ ਵਿੱਚ ਸਾਰਵਜਨਿਕ ਕੁੰਜੀ ਦੇ ਮੁਕਾਬਲੇ ਦਸਤਖ਼ਤ ਦੀ ਜਾਂਚ ਕਰਦੇ ਹਾਂ।

ਇਸ ਪੁਸ਼ਟੀਕਰਨ ਨੂੰ ਕਰਨ ਵਾਲਾ ਇੱਕ ਆਡੀਟਰ ਸਾਡੇ ਕੋਲੋਂ ਸਿਰਫ਼ ਰਸੀਦ ਹੀ ਲੈਣਾ ਚਾਹੀਦਾ ਹੈ। ਕਿਸੇ ਸੇਵਾ ਨੂੰ ਕਾਲ ਕਰਨ ਦੀ ਲੋੜ ਨਹੀਂ, ਕਿਸੇ ਕੁੰਜੀ ਡਾਇਰੈਕਟਰੀ ਤੋਂ ਪੁੱਛਤਾਛ ਕਰਨ ਦੀ ਲੋੜ ਨਹੀਂ, ਕਿਸੇ ਭਰੋਸੇ ਦੀ ਲੋੜ ਨਹੀਂ।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

ਤੁਸੀਂ ਦੇਖੋਗੇ `Receipt is valid: True`। ਏਜੰਟ ਨੇ ਆਪਣਾ ਪਹਿਲਾ ਗੁਪਤ ਲਿਪਿਬੱਧੀ ਤੌਰ 'ਤੇ ਸਾਇਨ ਕੀਤਾ ਆਡਿਟ ਰਿਕਾਰਡ ਤਿਆਰ ਕੀਤਾ ਹੈ।


## Section 2: Tamper with the receipt

ਦਰਜਾ ਪ੍ਰਾਪਤੀ ਦੀ ਸਾਰੀ ਮਕਸਦ ਇਹ ਹੈ ਕਿ ਉਹ ਤਬਦੀਲੀ-ਪ੍ਰਮਾਣਿਕ ਹੋਵਣ। ਆਓ ਇਸ ਨੂੰ ਸਾਬਿਤ ਕਰੀਏ।

ਅਸੀਂ ਰਸੀਦ ਦੇ ਬਿਲਕੁਲ ਇੱਕ ਅੱਖਰ ਨੂੰ ਬਦਲਾਂਗੇ ਅਤੇ ਵੈਰੀਫਿਕੇਸ਼ਨ ਦੇ ਨਾਕਾਮ ਹੋਣ ਨੂੰ ਦੇਖਾਂਗੇ।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ਕੀ ਹੁਣੇ ਹੋਇਆ?

ਜਦੋਂ ਅਸੀਂ `policy_id` ਨੂੰ ਬਦਲਿਆ, ਤਾਂ canonical ਬਾਈਟਸ ਬਦਲ ਗਏ। ਉਹਨਾਂ ਬਾਈਟਸ ਦਾ SHA-256 ਹੈਸ਼ ਬਦਲ ਗਿਆ। ਸਿਗਨੇਚਰ (ਜੋ ਅਸਲ ਹੈਸ਼ 'ਤੇ ਸੀ) ਹੁਣ ਨਵੇਂ ਹੈਸ਼ ਨਾਲ ਮੇਲ ਨਹੀਂ ਖਾਂਦੀ। ਵੈਰੀਫਿਕੇਸ਼ਨ ਸਹੀ ਤਰੀਕੇ ਨਾਲ `False` ਵਾਪਸ ਕਰਦਾ ਹੈ।

ਰਸੀਦ ਦੇ ਕਿਸੇ ਵੀ ਫੀਲਡ ਨੂੰ ਬਦਲਣ ਦਾ ਕੋਈ ਤਰੀਕਾ ਨਹੀਂ ਹੈ ਅਤੇ ਫਿਰ ਵੀ ਉਹ ਵੈਰੀਫਾਈ ਹੋਵੇਗੀ, ਜਦ ਤੱਕ ਹਮਲਾਵਰ ਕੋਲ ਪ੍ਰਾਈਵੇਟ ਕੀ ਨਾ ਹੋਵੇ। ਜਦ ਤੱਕ ਪ੍ਰਾਈਵੇਟ ਕੀ ਕੀਵਾਲਟ ਵਿੱਚ ਹੈ ਅਤੇ ਪਬਲਿਕ ਕੀ ਪ੍ਰਕਾਸ਼ਿਤ ਹੈ, ਚੇਡੀ-ਛਾਪੀ ਨੂੰ ਲੁਕਾਉਣਾ ਅਸੰਭਵ ਹੈ।

ਆਪਣੇ ਆਪ ਕੋਸ਼ਿਸ਼ ਕਰੋ: ਉੱਪਰ ਦਿੱਤੀ ਸੈੱਲ ਵਿੱਚ `tool_name` ਜਾਂ `agent_id` ਜਾਂ `timestamp` ਨੂੰ ਬਦਲੋ ਅਤੇ ਦੁਬਾਰਾ ਚਲਾਓ। ਹਰ ਬਦਲਾਅ ਗ਼ਲਤ ਰਸੀਦ ਬਣਾਉਂਦਾ ਹੈ।


## Section 3: ਚੇਨ ਰਸੀਦਾਂ ਨੂੰ ਇਕੱਠੇ ਜੋੜੋ

ਇੱਕ ਰਸੀਦ ਇੱਕ ਕਾਰਵਾਈ ਦੀ ਸੁਰੱਖਿਆ ਕਰਦੀ ਹੈ। ਜ਼ਿਆਦਾਤਰ ਏਜੰਟ ਕਈ ਕਾਰਵਾਈਆਂ ਕਰਦੇ ਹਨ। ਸਾਰੀ ਲੜੀ ਨੂੰ ਟੈਂਪਰ-ਸਬੂਤ ਬਣਾਉਣ ਲਈ, ਅਸੀਂ ਹਰ ਰਸੀਦ ਨੂੰ ਪਹਿਲਾਂ ਵਾਲੀ ਰਸੀਦ ਨਾਲ ਜੋੜਦੇ ਹਾਂ ਇਸ ਤਰ੍ਹਾਂ ਕਿ ਨਵੀਂ ਰਸੀਦ ਦੇ ਭਾਗ 'payload' ਵਿੱਚ ਪਿਛਲੀ ਰਸੀਦ ਦਾ ਹੈਸ਼ ਸ਼ਾਮਲ ਕੀਤਾ ਜਾਂਦਾ ਹੈ।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ਜੇ ਕੋਈ ਰਸੀਦ ਹਟਾ ਦਿੰਦਾ ਹੈ ਜਾਂ ਉਸਦੀ ਕ੍ਰਮਵਾਰਤਾ ਬਦਲ ਦਿੰਦਾ ਹੈ, ਤਾਂ ਲੜੀ ਉਸੇ ਥਾਂ ਤੋੜ ਜਾਂਦੀ ਹੈ। ਕਿਸੇ ਵੀ ਅਗਲੀ ਰਸੀਦ ਦੀ ਪੁਸ਼ਟੀ ਫੇਲ੍ਹ ਹੋ ਜਾਂਦੀ ਹੈ ਕਿਉਂਕਿ ਉਸਦਾ `previous_receipt_hash` ਹੁਣ ਇਸਦੇ ਪਿਛਲੇ ਰਸੀਦ ਦੇ ਅਸਲ ਹੈਸ਼ ਨਾਲ ਮੇਲ ਨਹੀਂ ਖਾਂਦਾ।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ਹੁਣ ਵਿਚਕਾਰਲੇ ਰਸੀਦ ਨਾਲ ਛੇੜਛਾੜ ਕਰਕੇ ਚੇਨ ਨੂੰ ਤੋੜੋ ਅਤੇ ਮੁੜ-ਸत्यਾਪਨ ਕਰੋ। ਛੇੜ-ਛਾੜ ਕੀਤੀ ਰਸੀਦ ਆਪਣੀ ਦਸਤਖਤ ਚੈੱਕ ਵਿੱਚ ਫੇਲ ਹੋ ਜਾਂਦੀ ਹੈ, ਅਤੇ ਅਗਲੀ ਰਸੀਦ ਆਪਣੀ ਚੇਨ-ਲਿੰਕ ਚੈੱਕ ਵਿੱਚ ਫੇਲ ਹੋ ਜਾਂਦੀ ਹੈ (ਕਿਉਂਕਿ ਇਸਦਾ `previous_receipt_hash` ਹੁਣ ਤਬਦੀਲ ਕੀਤੀ ਗਈ ਵਿਚਕਾਰਲੀ ਰਸੀਦ ਦੇ ਹੈਸ਼ ਨਾਲ ਮੇਲ ਨਹੀਂ ਖਾਂਦਾ)।


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ਰੀਸੀਟ 0 ਅਜੇ ਵੀ ਸੱਚੀ ਹੈ (ਇਹ ਸੋਧਿਆ ਨਹੀਂ ਗਿਆ ਅਤੇ ਇਸ ਦਾ ਕੋਈ ਪੂਰਵਜ ਨਹੀਂ ਜਿਸ 'ਤੇ ਇਹ منحصر ਹੋਵੇ)। ਰੀਸੀਟ 1 ਆਪਣੀ ਸਾਈਨਚੇਕ ਵਿੱਚ ਅਸਫਲ ਰਹਿੰਦੀ ਹੈ ਕਿਉਂਕਿ ਅਸੀਂ `tool_args_hash` ਨੂੰ ਬਦਲ ਦਿੱਤਾ ਹੈ। ਰੀਸੀਟ 2 ਆਪਣੀ ਚੇਨ-ਲਿੰਕ ਚੈੱਕ ਵਿੱਚ ਅਸਫਲ ਰਹਿੰਦੀ ਹੈ ਕਿਉਂਕਿ ਇਸ ਦਾ `previous_receipt_hash` ਮੂਲ (ਹੁਣ-ਸੋਧਿਆ ਗਿਆ) ਰੀਸੀਟ 1 ਦੇ ਖਿਲਾਫ ਗਣਨਾ ਕੀਤਾ ਗਿਆ ਸੀ।

ਜੇਕਰ ਹਮਲਾਵਰ ਸੋਧਿਆ ਹੋਇਆ ਰੀਸੀਟ 1 ਨੂੰ ਮੁੜ-ਸਾਈਨ ਕਰਦਾ ਵੀ ਹੈ (ਜੋ ਉਹ ਨਿੱਜੀ ਕੁੰਜੀ ਬਿਨਾਂ ਨਹੀਂ ਕਰ ਸਕਦਾ), ਤਾਂ ਵੀ ਰੀਸੀਟ 2 ਵਿੱਚ ਚੇਨ-ਲਿੰਕ ਅਸਮਨਵਤਾ ਛੇੜਖਾਨੀ ਨੂੰ ਬਿਆਨ ਕਰੇਗੀ। ਬਦਲਾਅ ਨੂੰ ਛੁਪਾਉਣ ਲਈ, ਹਮਲਾਵਰ ਨੂੰ ਸੋਧ ਬਿੰਦੂ ਤੋਂ ਆਗੇ ਹਰ ਇੱਕ ਰੀਸੀਟ ਨੂੰ ਮੁੜ-ਸਾਈਨ ਕਰਨਾ ਪਵੇਗਾ, ਜਿਸ ਲਈ ਨਿੱਜੀ ਕੁੰਜੀ ਦੀ ਮਲਕੀਅਤ ਜ਼ਰੂਰੀ ਹੈ।


## Section 4: ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਨੂੰ ਰਸੀਦ ਸਾਈਨਿੰਗ ਨਾਲ ਘੇਰੋ

ਵਾਸਤਵਿਕ ਤਾਇਨਾਤੀ ਵਿੱਚ, ਤੁਸੀਂ ਹਰ ਏਜੰਟ ਲੇਖਕ ਤੋਂ ਇਹ ਯਾਦ ਰੱਖਣ ਦੀ ਉਮੀਦ ਨਹੀਂ ਕਰਦੇ ਕਿ ਉਹ `make_receipt` ਨੂੰ ਕਾਲ ਕਰੇ। ਤੁਸੀਂ ਚਾਹੁੰਦੇ ਹੋ ਕਿ ਹਰ ਟੂਲ ਕਾਲ ਲਈ ਰਸੀਦ ਸਾਈਨਿੰਗ ਆਪ-ਆਪ ਹੋ ਜਾਵੇ।

ਇੱਥੇ ਸਭ ਤੋਂ ਸਧਾਰਣ ਨਮੂਨਾ ਦਿੱਤਾ ਗਿਆ ਹੈ: ਇੱਕ ਰੈਪਰ ਕਲਾਸ ਜੋ ਕਿਸੇ ਵੀ ਕਾਲਯੋਗ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਲੈਂਦੀ ਹੈ ਅਤੇ ਇਸਦਾ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਕਰਨ ਵਾਲਾ ਸੰਸਕਰਨ ਵਾਪਸ ਕਰਦੀ ਹੈ। ਇਹ ਕਿਸੇ ਵੀ ਏਜੰਟ ਫਰੇਮਵਰਕ, ਜਿਸ ਵਿੱਚ Microsoft Agent Framework (`agent_framework.azure`) ਵੀ ਸ਼ਾਮਿਲ ਹੈ, ਲਈ ਅਨੁਕੂਲ ਹੈ।

ਜੇ ਤੁਹਾਡੇ ਕੋਲ Azure AI Foundry ਪ੍ਰాజੈਕਟ ਸੈੱਟਅਪ ਨਹੀਂ ਹੈ, ਤਾਂ ਹੇਠਾਂ ਦਿੱਤਾ ਗਿਆ ਸਥਾਨਕ ਮੌਕ ਫਿਰ ਵੀ ਇਸ ਨਮੂਨੇ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਇੰਟੀਗ੍ਰੇਟ ਕਰਨਾ

ਉੱਪਰ ਦਿੱਤਾ `ReceiptedTool` ਰੈੱਪਰ ਫਰੇਮਵਰਕ-ਅਗਨੋਸਟਿਕ ਹੈ। ਇਸਨੂੰ ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਬਣਾਏ ਏਜੰਟ ਦੇ ਅੰਦਰ ਵਰਤਣ ਲਈ, ਵ੍ਰੈਪ ਕੀਤੇ ਫੰਕਸ਼ਨ ਨੂੰ ਟੂਲ ਵਜੋਂ ਰਜਿਸਟਰ ਕਰੋ। ਇਕ ਖਾਕਾ (ਤੁਸੀਂ ਮੌਕ ਨੂੰ ਇੱਕ ਅਸਲੀ Azure AI Foundry ਟੂਲ ਰਜਿਸਟ੍ਰੇਸ਼ਨ ਨਾਲ ਬਦਲੋਗੇ):

```python
# ਇੰਟਿਗ੍ਰੇਸ਼ਨ ਆਕਾਰ ਦਿਖਾਉਂਦਾ ਕੂਪਸੂਡੋ ਕੋਡ।
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# ਪ੍ਰੋਵਾਈਡਰ = AzureAIProjectAgentProvider(ਕ੍ਰੈਡੈਂਸ਼ੀਅਲ=AzureCliCredential())
# ਏਜੰਟ = ਪ੍ਰੋਵਾਈਡਰ.create_agent(
#     ਹਦਾਇਤਾਂ="ਤੁਸੀਂ ਇੱਕ ਕੋਂਟੋਸੋ ਟ੍ਰੈਵਲ ਏਜੰਟ ਹੋ ..."
#     ਟੂਲ = [receipted_lookup],   # ਲਪੇਟਿਆ ਹੋਇਆ ਟੂਲ, ਕੱਚਾ ਫੰਕਸ਼ਨ ਨਹੀਂ
# )
# ਜਵਾਬ = ਏਜੰਟ.run("ਜੂਨ ਵਿੱਚ ਸਿਡਨੀ ਤੋਂ ਲਾਸ ਐੰਜਲਸ ਲਈ ਉਡਾਣਾਂ ਲੱਭੋ।")
#
# # ਚਲਾਉਣ ਤੋਂ ਬਾਅਦ, ਹਰ ਟੂਲ ਕਾਲ ਤੇ ਏਜੰਟ ਨੇ ਇੱਕ ਸਾਈਨ ਕੀਤੀ ਰਸੀਦ ਹੁੰਦੀ ਹੈ:
# ਆਡਿਟ ਚੇਨ = receipted_lookup.receipts
```

ਏਜੰਟ ਫਰੇਮਵਰਕ ਨੂੰ ਰਸੀਦਾਂ ਬਾਰੇ ਕੁਝ ਜਾਣਣਾ ਲਾਜ਼ਮੀ ਨਹੀਂ ਹੈ। ਰਸੀਦ ਸਾਇਨਿੰਗ ਟੂਲ ਦੇ ਆਲੇ ਦੁਆਲੇ ਰੈੱਪ ਕੀਤੀ ਜਾਂਦੀ ਹੈ, ਫਰੇਮਵਰਕ ਵਿੱਚ ਫਿੱਟ ਨਹੀਂ ਕੀਤੀ ਜਾਂਦੀ। ਇਹ ਹੈ ਕਿ ਤੁਸੀਂ ਮੌਜੂਦਾ ਏਜੰਟ ਕੋਡ ਵਿੱਚ ਪ੍ਰੋਵੇਨੈਂਸ ਕਿਵੇਂ ਸ਼ਾਮਲ ਕਰਦੇ ਹੋ ਬਿਨਾਂ ਏਜੰਟ ਨੂੰ ਮੁੜ ਲਿਖੇ।


## ਸਮਰੀ ਅਤੇ ਸਟ੍ਰੈਚ ਚੈਲेंज

ਤੁਹਾਡੇ ਕੋਲ ਹੈ:

- ਇੱਕ Ed25519 ਕੁੰਜੀ ਜੋੜੀ ਤਿਆਰ ਕੀਤੀ।
- ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ رسید ਬਣਾਈ ਅਤੇ ਸਾਈਨ ਕੀਤੀ।
- ਸਿਰਫ਼ ਪਬਲਿਕ ਕੀ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਫਲਾਈਨ ਰਸੀਦ ਦੀ ਵੈਰੀਫਿਕੇਸ਼ਨ ਕੀਤੀ।
- ਇੱਕ ਰਸੀਦ ਨਾਲ ਛੇੜਛਾੜ ਕੀਤੀ ਅਤੇ ਵੈਰੀਫਿਕੇਸ਼ਨ ਫੇਲ ਹੋਣ ਨੂੰ ਦੇਖਿਆ।
- ਤਿੰਨ ਰਸੀਦਾਂ ਦੀ ਇੱਕ ਹੈਸ਼-ਚੇਨਡ ਲੜੀ ਬਣਾਈ।
- ਲੜੀ ਦੇ ਵਿਚਕਾਰ ਛੇੜਛਾੜ ਕੀਤੀ ਅਤੇ ਦੋਹਾਂ ਦਸਤਖਤ ਅਸਫਲਤਾ ਅਤੇ ਲੜੀ-ਲਿੰਕ ਅਸਫਲਤਾ ਨੂੰ ਦੇਖਿਆ।
- ਆਟੋਮੈਟਿਕ ਰਸੀਦ ਸਾਈਨਿੰਗ ਨਾਲ ਇੱਕ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਰੈਪ ਕੀਤਾ।

**ਸਟ੍ਰੈਚ ਚੈਲेंज।** ਰਸੀਦ ਸਕੀਮਾ ਵਿੱਚ `request_id` ਖੇਤਰ ਵਧਾਓ (ਵੰਡੇ ਜਾਂਦੇ ਟਰੇਸਿੰਗ ਲਈ ਇੱਕ UUID)। `make_receipt` ਨੂੰ ਇਸ ਵਿੱਚ ਸ਼ਾਮਿਲ ਕਰਨ ਲਈ ਅਪਡੇਟ ਕਰੋ, ਅਤੇ ਪੁਸ਼ਟੀ ਕਰੋ ਕਿ ਰਸੀਦਾਂ ਅਜੇ ਵੀ ਆਖਰੀ ਤੱਕ ਵੈਰੀਫਾਈ ਹੁੰਦੀਆਂ ਹਨ। ਫਿਰ ਸਾਈਨਿੰਗ ਤੋਂ ਬਾਅਦ ਖੇਤਰ ਵਿੱਚ ਬਦਲਾਅ ਕਰੋ ਅਤੇ ਪੁਸ਼ਟੀ ਕਰੋ ਕਿ ਵੈਰੀਫਿਕੇਸ਼ਨ ਫੇਲ ਹੁੰਦੀ ਹੈ। ਇਹ ਤੁਹਾਨੂੰ ਜ਼ਬਤ ਕਰਦਾ ਹੈ ਕਿ canonical ਐਨਕੋਡਿੰਗ ਦਾ ਹਰ ਬਾਈਟ ਦਸਤਖਤ ਵਿੱਚ ਕਿਵੇਂ ਯੋਗਦਾਨ ਪਾਂਦਾ ਹੈ।

**ਜ਼ਰੂਰੀ ਸੀਮਾ।** ਰਸੀਦ ਤਿੰਨ ਗੱਲਾਂ ਨੂੰ ਸਾਬਤ ਕਰਦੀਆਂ ਹਨ ਅਤੇ ਸਿਰਫ਼ ਤਿੰਨ ਗੱਲਾਂ: attribution (ਇਸ ਕੁੰਜੀ ਨੇ ਇਸ ਸਮੱਗਰੀ 'ਤੇ ਦਸਤਖਤ ਕੀਤਾ), ਸੁਰੱਖਿਆ (ਸਾਈਨਿੰਗ ਤੋਂ ਬਾਅਦ ਸਮੱਗਰੀ ਵਿੱਚ ਕੋਈ ਬਦਲਾਅ ਨਹੀਂ ਹੋਇਆ), ਅਤੇ ਆਰਡਰਿੰਗ (ਇਹ ਰਸੀਦ ਉਸ ਰਸੀਦ ਤੋਂ ਬਾਅਦ ਆਈ)। ਇਹ ਸਾਬਤ ਨਹੀਂ ਕਰਦੀਆਂ ਕਿ ਏਜੰਟ ਦੀ ਕਾਰਵਾਈ ਸਹੀ ਸੀ, `policy_id` ਵਿੱਚ ਦਿੱਤੀ ਨੀਤੀ ਦਾ ਸਹੀ ਤਰੀਕੇ ਨਾਲ ਮੁੱਲਾਂਕਣ ਹੋਇਆ, ਜਾਂ ਏਜੰਟ ਨੇ ਹਰ ਨਿਯਮ ਦਾ ਪਾਲਣ ਕੀਤਾ। ਰਸੀਦਾਂ ਇੱਕ ਬੁਨਿਆਦ ਹਨ। ਸ਼ਾਸਨ ਉਹ ਪ੍ਰਣਾਲੀ ਹੈ ਜੋ ਤੁਸੀਂ ਇਸ ਦੇ ਉਪਰ ਬਣਾਉਂਦੇ ਹੋ।

ਉਸ ਸੀਮਾ ਨੂੰ ਧਿਆਨ ਵਿੱਚ ਰੱਖਦੇ ਹੋਏ ਲੈਸਨ README ਨੂੰ ਫਿਰ ਤੋਂ ਪੜ੍ਹੋ। ਰਸੀਦਾਂ ਨਾਲ ਸਭ ਤੋਂ ਆਮ ਗਲਤੀ ਟੀਮਾਂ ਕਰਦੀਆਂ ਹਨ ਉਹ ਹੈ "ਸਾਡੇ ਕੋਲ ਰਸੀਦਾਂ ਹਨ" ਨੂੰ "ਸਾਡੇ 'ਤੇ ਸ਼ਾਸਨ ਹੁੰਦਾ ਹੈ" ਸਮਝਣਾ। ਇਹ ਸਹੀ ਨਹੀਂ ਹੈ। ਰਸੀਦਾਂ ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਨੂੰ ਆਡੀਟ ਕਰਨਯੋਗ ਬਣਾਉਂਦੀਆਂ ਹਨ। ਇਹ ਇਸਨੂੰ ਸਹੀ ਨਹੀਂ ਬਣਾਉਂਦੀਆਂ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
